# Limpeza e Preparação dos Dados 🏗️ 🎲

<pre>Algumas organizações oferecem bibliotecas para facilitar a criação de consultas, integrando-as ao nosso código.</pre>

<pre>A presente análise será realizada com base nos dados dos países pertencentes à <a href="https://databank.worldbank.org/source/world-development-indicators">base de dados do Banco Mundial</a>, que possuem a informação de PIB <i>per capita</i> (current US$) disponível para o ano de 2019.</pre>

## Explorando os dados econômicos do Banco Mundial

<pre>O Banco Mundial oferece biblioteca em Python para consultar seus bancos de dados.</pre>

👉 mais detalhes, consulte o pacote <a href='https://pypi.org/project/wbgapi/'>wbgapi</a> e sua <a href='https://blogs.worldbank.org/opendata/introducing-wbgapi-new-python-package-accessing-world-bank-data'>documentação</a>.

<pre>Vamos instalar e importar as bibliotecas que serão utilizadas neste caderno.</pre>

In [1]:
!pip install wbgapi --quiet
!pip install pandas --quiet

<pre>Execute as 2 células abaixo para montar o DataFrame <b>df</b>, com dados econômicos dos países.</pre>

In [2]:
import wbgapi as wb
import pandas as pd

In [3]:
economy_info = wb.economy.info()

countries = [pais for pais in economy_info.items if not pais.get('aggregate')]

countries_dict = {"country_code": [country.get("id") for country in countries],
                  "country_name": [country.get("value") for country in countries]}

countries_df = pd.DataFrame(countries_dict).set_index("country_code")

indicators_df = wb.data.DataFrame(series = ['NY.GDP.PCAP.CD', 'NE.GDI.FTOT.CD', 'SP.POP.TOTL', 
                                            'SL.TLF.ADVN.ZS',  'AG.LND.ARBL.HA.PC', 'NV.IND.TOTL.ZS', 
                                            'SE.XPD.TOTL.GD.ZS'],
                                  economy = countries_df.index,
                                  time = 2019)

df = countries_df.join(indicators_df)

df.head()

,country_name,AG.LND.ARBL.HA.PC,NE.GDI.FTOT.CD,NV.IND.TOTL.ZS,NY.GDP.PCAP.CD,SE.XPD.TOTL.GD.ZS,SL.TLF.ADVN.ZS,SP.POP.TOTL
country_code,,,,,,,,
ABW,Aruba,0.018315,NaN,11.372037,31096.205074,4.435043,NaN,109203.0
AFG,Afghanistan,0.205726,NaN,14.058112,496.602504,NaN,NaN,37856121.0
AGO,Angola,0.165649,1.255763e+10,49.760922,2189.855714,2.073064,91.473,32375632.0
ALB,Albania,0.213721,3.834691e+09,22.965531,5460.428237,3.916240,75.465,2854191.0
AND,Andorra,0.010345,NaN,11.699818,41257.804585,3.150610,NaN,76474.0


<pre>Segue o dicionário de dados do DataFrame acima. Isto é, o significado das siglas das colunas do DataFrame, conforme o Banco Mundial:
    - NY.GDP.PCAP.CD: GDP per capita (current US$);
    - NE.GDI.FTOT.CD: Gross fixed capital formation (current US$);
    - SP.POP.TOTL: Population, total;
    - SL.TLF.ADVN.ZS: Labor force with advanced education (% of total working-age population with advanced education);
    - AG.LND.ARBL.HA.PC: Arable land (hectares per person);
    - NV.IND.TOTL.ZS: Industry (including construction), value added (% of GDP);
    - SE.XPD.TOTL.GD.ZS: Government expenditure on education, total (% of GDP).</pre>

<pre>Agora, o que explica o nível de pobreza/riqueza de determinada sociedade? Existem diversas teorias que tentam explicar essa pergunta.</pre>

<pre>Vamos analisar dados dos países do mundo, suas características de acordo com seu nível de riqueza e tentar entender melhor essa questão.</pre>

<pre>Comece renomeando algumas variáveis, isto é, o cabeçalho de algumas colunas, da seguinte forma:
    - coluna <b>NY.GDP.PCAP.CD</b>: renomear para <b>GDP_PC</b>;
    - coluna <b>SL.TLF.ADVN.ZS</b>: renomear para <b>forca_trab_educ</b>;
    - coluna <b>AG.LND.ARBL.HA.PC</b>: renomear para <b>arable_land</b>;
    - coluna <b>NV.IND.TOTL.ZS</b>: renomear para <b>industria_PERCPIB</b>;
    - coluna <b>SE.XPD.TOTL.GD.ZS</b>: renomear para <b>gasto_educ_PERCPIB</b>;
    - coluna <b>NE.GDI.FTOT.CD</b>: renomear para <b>FBKF</b>;
    - coluna <b>SP.POP.TOTL</b>: renomear para <b>populacao.</b></pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df = df.rename(columns={
    "NY.GDP.PCAP.CD": "GDP_PC",
    "SL.TLF.ADVN.ZS" : "forca_trab_educ",
    "AG.LND.ARBL.HA.PC": "arable_land",
    "NV.IND.TOTL.ZS": "industria_PERCPIB",
    "SE.XPD.TOTL.GD.ZS": "gasto_educ_PERCPIB",
    "NE.GDI.FTOT.CD": "FBKF",
    "SP.POP.TOTL": "populacao"})
```

<br/>

</details>

<pre>Tente compreender, de forma geral, as informações do dataframe <b>df</b>.</pre>

👉 dica: use o método `head` do DataFrame `df`.

<details>
  <summary>Resposta</summary>

<br/>

```python
df.head()
```

<br/>

</details>

<pre>Quantos países e quantas variáveis temos no nosso banco de dados?</pre>

👉 dica: use o atributo `shape` do DataFrame `df`, que retornará $(x, y)$, em que:

    - $x$ corresponde ao número de linhas/países; e 
    - $y$ corresponde o número de colunas/variáveis.

<details>
  <summary>Resposta</summary>

<br/>

```python
df.shape
```

<br/>

</details>

<pre>Agora, inspecione as principais estatísticas descritivas dos dados com o método <b>describe</b> do DataFrame <b>df</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df.describe()
```

<br/>

</details>

<pre>Por que você acha que vieram informações de 7 variáveis, quando sabemos ter 8?</pre>

👉 dica: use o método `info` do DataFrame `df` para saber os tipos de dados de cada variável.

<details>
  <summary>Resposta</summary>

<br/>

```python
df.info()
```

<br/>

</details>

<pre>O tipo de dado <b>object</b> é usado para representar cadeias de caracteres em DataFrames, quando essas cadeias de caracteres não têm um tipo de dados homogêneo ou formam uma combinação de diferentes tipos de dados, como números e cadeias de caracteres.</pre>

<pre>Por padrão, o método <b>describe</b> de um DataFrame apenas apresenta as estatísticas descritivas dos tipos de dados que representam números.</pre>

<pre>Para que o método <b>describe</b> de um DataFrame apresente as estatísticas descritivas dos tipos de dados que representam cadeias de caracteres, deve-se utilizar o valor <b>'all'</b> para o parâmetro <b>include</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
df.describe(include='all')
```

<br/>

</details>

🚨 Perceba que há valores ausentes em algumas estatísticas descritivas de cada uma das variáveis/colunas 🚨

### Dados Ausentes

<pre>Da nossa aula, sabemos que algumas representações comuns de valores ausentes são:
    - NaN 
    - 999 
    - .
    - ..
    - ?
    - faltante
    - " "</pre>

<pre>Identifique qual a representação padrão do DataFrame <b>df</b> para os valores ausentes.</pre>

<pre>O comando abaixo apresentará a proporção de cada um dos valores na coluna/variável <b>FBKF</b>.</pre>

In [ ]:
df['FBKF'].value_counts(normalize=True, dropna=False).map(lambda x: f"{x:.2%}")

<pre>E agora? O que fazer com os dados ausentes?</pre>

#### Preenchendo Dados Ausentes

<pre>Como vimos na aula, para os dados ausentes, podemos dar alguns tratamentos:
    - Excluir a variável inteira; 
    - Imputar valores (média, mediana, etc);
    - Substituir (manualmente) por valores mais significativos.</pre>

<pre>Por exemplo, substitua os valores ausentes da coluna/variável <b>FBKF</b>, pela média dos valores disponíveis dessa variável.</pre>

👉 dica: use o método `fillna` na coluna/variável `FBKF` e passe o valor que deseja imputar como parâmetro

<details>
  <summary>Resposta</summary>

<br/>

```python
df['FBKF'] = df['FBKF'].fillna(df['FBKF'].mean())
```

<br/>

</details>

<pre>Agora, ao apresentar a proporção de cada um dos valores na coluna/variável <b>FBKF</b>, veja que não há mais valores ausentes.</pre>

In [ ]:
df['FBKF'].value_counts(normalize=True, dropna=False).map(lambda x: f"{x:.2%}")

<pre>Agora, assim como fizemos com a coluna/variável <b>FBKF</b>, inspecione as outras colunas/variáveis, uma a uma, e, caso exista valores ausentes na coluna/variável sendo inspecionada, os substitua por outro valores, conforme as diferentes estratégias apresentadas em sala de aula.</pre>